In [24]:
# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================
import pandas as pd
from datetime import datetime

In [25]:
# ============================================================
# STEP 2: LOAD ALL FILES
# ============================================================
orders = pd.read_csv('DATA/orders.csv')
nodes = pd.read_csv('DATA/nodes.csv')  
picking = pd.read_csv('DATA/picking.csv')
shipments = pd.read_csv('DATA/shipments.csv')
inventory = pd.read_csv('DATA/inventory.csv')
costs = pd.read_csv('DATA/costs.csv')

# ============================================================
# STEP 3: INSPECT FIRST (always do this before cleaning)
# ============================================================
def inspect(df, name):
    print(f"\n===== {name} =====")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Missing values:\n{df.isnull().sum()}")
    print(f"Duplicates: {df.duplicated().sum()}")
    print(f"Data types:\n{df.dtypes}")
    print(df.head(3))

inspect(orders, "ORDERS")
inspect(nodes, "NODES")
inspect(picking, "PICKING")
inspect(shipments, "SHIPMENTS")
inspect(inventory, "INVENTORY")
inspect(costs, "COSTS OF NODE")


===== ORDERS =====
Shape: (40000, 10)
Columns: ['order_id', 'customer_id', 'node_id', 'order_date', 'promised_delivery', 'actual_delivery', 'order_status', 'order_value', 'order_priority', 'order_type']
Missing values:
order_id               0
customer_id            0
node_id                0
order_date             0
promised_delivery      0
actual_delivery      535
order_status           0
order_value            0
order_priority         0
order_type             0
dtype: int64
Duplicates: 0
Data types:
order_id               int64
customer_id            int64
node_id                int64
order_date               str
promised_delivery        str
actual_delivery          str
order_status             str
order_value          float64
order_priority           str
order_type               str
dtype: object
   order_id  customer_id  node_id           order_date    promised_delivery  \
0    100001        17529        6  2025-11-18 13:04:00  2025-11-19 13:44:30   
1    100002        11463     

In [29]:
# STEP 5: FIX DATA TYPES
# ============================================================
# Convert date columns to datetime
orders['order_date'] = pd.to_datetime(orders['order_date'], dayfirst=True, errors='coerce')
orders['promised_delivery'] = pd.to_datetime(orders['promised_delivery'],dayfirst=True, errors='coerce')
orders['actual_delivery'] = pd.to_datetime(orders['actual_delivery'],dayfirst=True, errors='coerce')
picking['pick_start'] = pd.to_datetime(picking['pick_start'],dayfirst=True, errors='coerce')
picking['pick_end'] = pd.to_datetime(picking['pick_end'],dayfirst=True, errors='coerce')
shipments['dispatch_date'] = pd.to_datetime(shipments['dispatch_date'], dayfirst=True,errors='coerce')
shipments['expected_arrival'] = pd.to_datetime(shipments['expected_arrival'],dayfirst=True, errors='coerce')
shipments['actual_arrival'] = pd.to_datetime(shipments['actual_arrival'],dayfirst=True, errors='coerce')
inventory['last_audit_date'] = pd.to_datetime(inventory['last_audit_date'], dayfirst=True, errors='coerce')
costs['cost_date'] = pd.to_datetime(costs['cost_date'],dayfirst=True,  errors='coerce')

print(f"Data types of orders:\n{orders.dtypes}")
print(f"Data types of picking:\n{picking.dtypes}")
print(f"Data types of shipments:\n{shipments.dtypes}")
print(f"Data types of inventory:\n{inventory.dtypes}")
print(f"Data types of costs:\n{costs.dtypes}")

Data types of orders:
order_id                      int64
customer_id                   int64
node_id                       int64
order_date           datetime64[us]
promised_delivery    datetime64[us]
actual_delivery      datetime64[ns]
order_status                    str
order_value                 float64
order_priority                  str
order_type                      str
dtype: object
Data types of picking:
pick_id                  int64
node_id                  int64
worker_id                  str
order_id                 int64
pick_start      datetime64[ns]
pick_end        datetime64[ns]
units_picked             int64
errors                   int64
dtype: object
Data types of shipments:
shipment_id                  int64
from_node_id                 int64
to_node_id                   int64
shipment_type                  str
carrier                        str
dispatch_date       datetime64[us]
expected_arrival    datetime64[us]
actual_arrival      datetime64[ns]
status        

In [30]:
# STEP 6: CHECKING MISSING VALUE 
# ============================================================
print("Only the canceled order dates are missing : \n", orders.groupby('order_status')['actual_delivery'].apply(lambda x: x.isnull().sum()))


Only the canceled order dates are missing : 
 order_status
Cancelled    535
Delivered      0
Returned       0
Name: actual_delivery, dtype: int64


In [31]:
# STEP 6: ADDING COLUMN
# ============================================================
def format_duration(x):
    total_minutes = x.total_seconds() / 60

    if total_minutes >= 1440:
        return f"{total_minutes / 1440:g} days"
    elif total_minutes >= 60:
        return f"{total_minutes / 60:g} hr"
    else:
        return f"{total_minutes:g} min"

orders['total_order_delivery_days'] = (orders['actual_delivery'] - orders['order_date']).apply(format_duration)
orders['order_delivery_delay'] = (orders['actual_delivery'] - orders['promised_delivery']).apply(format_duration)

duration = picking['pick_end'] - picking['pick_start']
picking['picking_delivery_time'] = duration.apply(lambda x:f"{x.total_seconds() / 86400:.1f} days"if x.total_seconds() >= 86400
        else (f"{x.total_seconds() / 3600:.1f} hr"
              if x.total_seconds() >= 3600
              else f"{x.total_seconds() / 60:.0f} min"))

shipments['total_shipment_delivery_days'] = (shipments['actual_arrival'] - shipments['dispatch_date']).apply(format_duration)
shipments['shipment_delivery_delay'] = (shipments['actual_arrival'] - shipments['expected_arrival']).apply(format_duration)

In [32]:
print(f"orders Columns: {list(orders.columns)}")
orders[['order_id','order_date','promised_delivery','actual_delivery','total_order_delivery_days','order_delivery_delay']].head(3)

orders Columns: ['order_id', 'customer_id', 'node_id', 'order_date', 'promised_delivery', 'actual_delivery', 'order_status', 'order_value', 'order_priority', 'order_type', 'total_order_delivery_days', 'order_delivery_delay']


,order_id,order_date,promised_delivery,actual_delivery,total_order_delivery_days,order_delivery_delay
0,100001,2025-11-18 13:04:00,2025-11-19 13:44:30,2025-11-19 11:22:38.692052604,22.3107 hr,-141.855 min
1,100002,2025-12-31 16:44:00,2026-01-01 17:06:30,2026-01-01 12:38:40.968110034,19.9114 hr,-267.817 min
2,100003,2026-02-04 19:56:00,2026-02-05 08:18:30,2026-02-05 06:05:26.293742306,10.1573 hr,-133.062 min


In [33]:
print(f"picking Columns: {list(picking.columns)}")
picking[['pick_start','pick_end','picking_delivery_time']].head(3)

picking Columns: ['pick_id', 'node_id', 'worker_id', 'order_id', 'pick_start', 'pick_end', 'units_picked', 'errors', 'picking_delivery_time']


,pick_start,pick_end,picking_delivery_time
0,2025-11-18 13:46:40.205712930,2025-11-18 13:58:18.349135962,12 min
1,2025-12-31 17:34:12.621138120,2025-12-31 17:42:55.887198774,9 min
2,2026-02-04 20:45:02.931563064,2026-02-04 20:53:36.085129518,9 min


In [34]:
print(f"shipments Columns: {list(shipments.columns)}")
shipments[['dispatch_date','expected_arrival','actual_arrival','total_shipment_delivery_days','shipment_delivery_delay']].head(3)

shipments Columns: ['shipment_id', 'from_node_id', 'to_node_id', 'shipment_type', 'carrier', 'dispatch_date', 'expected_arrival', 'actual_arrival', 'status', 'shipment_value', 'total_shipment_delivery_days', 'shipment_delivery_delay']


,dispatch_date,expected_arrival,actual_arrival,total_shipment_delivery_days,shipment_delivery_delay
0,2026-01-26 20:00:00,2026-01-26 22:12:00,2026-01-27 01:29:27.812981878,5.49106 hr,3.29106 hr
1,2026-02-12 14:00:00,2026-02-12 16:12:00,2026-02-12 21:31:51.706145452,7.53103 hr,5.33103 hr
2,2026-03-07 18:00:00,2026-03-09 02:00:00,2026-03-09 05:58:17.423953525,1.49881 days,3.97151 hr


In [35]:
#Connecting Python script to PostgreSQL

In [36]:
from sqlalchemy import create_engine

# Step 1: Connect to PostgreSQL
username = "postgres"      # default user
password = "0000" # the password you set during installation
host = "localhost"         # if running locally
port = "5432"              # default PostgreSQL port
database = "Ecommerce_Delivery"    # the database you created in pgAdmin

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Step 2: Upload all 5 DataFrames
orders.to_sql("orders",engine,if_exists="replace",index=False)
nodes.to_sql("nodes",engine,if_exists="replace",index=False)
picking.to_sql("picking",engine,if_exists="replace",index=False)
shipments.to_sql("shipments",engine,if_exists="replace",index=False)
inventory.to_sql("inventory",engine,if_exists="replace",index=False)
costs.to_sql("costs",engine,if_exists="replace",index=False)

print("All 5 tables successfully loaded into PostgreSQL!")

All 5 tables successfully loaded into PostgreSQL!


In [37]:
print(f" orders Columns orders : {list(orders.columns)}\n")
print(f" nodes Columns: {list(nodes.columns)}\n")
print(f" picking Columns: {list(picking.columns)}\n")
print(f" shipments Columns: {list(shipments.columns)}\n")
print(f" inventory Columns: {list(inventory.columns)}\n")
print(f" costs Columns: {list(costs.columns)}\n")

 orders Columns orders : ['order_id', 'customer_id', 'node_id', 'order_date', 'promised_delivery', 'actual_delivery', 'order_status', 'order_value', 'order_priority', 'order_type', 'total_order_delivery_days', 'order_delivery_delay']

 nodes Columns: ['node_id', 'node_name', 'node_type', 'city', 'capacity_units']

 picking Columns: ['pick_id', 'node_id', 'worker_id', 'order_id', 'pick_start', 'pick_end', 'units_picked', 'errors', 'picking_delivery_time']

 shipments Columns: ['shipment_id', 'from_node_id', 'to_node_id', 'shipment_type', 'carrier', 'dispatch_date', 'expected_arrival', 'actual_arrival', 'status', 'shipment_value', 'total_shipment_delivery_days', 'shipment_delivery_delay']

 inventory Columns: ['inventory_id', 'node_id', 'product_id', 'system_qty', 'physical_qty', 'last_audit_date']

 costs Columns: ['cost_id', 'node_id', 'cost_date', 'cost_type', 'amount']

